In [ ]:
# DataPilot AI - Next-Gen Conversational Analytics & Auto-Prep Studio
# Colab install cell:  !pip install -q "gradio>=4.44" plotly openpyxl xlrd pandas numpy

import os, re, inspect, tempfile, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr

warnings.filterwarnings("ignore")
ID_RE = re.compile(r"(^|[_\s])(id|code|key|sku|uuid)($|[_\s])|id$", re.I)
MAX_MB = 200

# ---------------- Cross-Version Gradio Chatbot Compatibility ----------------
_CHAT_PARAMS = inspect.signature(gr.Chatbot.__init__).parameters
CHAT_SUPPORTS_TYPE = "type" in _CHAT_PARAMS
# Gradio 6+ removed `type` and uses OpenAI-style message dicts by default;
# Gradio 4/5 with `type` supports `type="messages"`; older Gradio 4 uses tuple pairs.
CHAT_USE_MESSAGES = True if CHAT_SUPPORTS_TYPE else (int(gr.__version__.split(".")[0]) >= 6)

def format_chat_init(welcome_markdown):
    if CHAT_USE_MESSAGES:
        return [{"role": "assistant", "content": welcome_markdown}]
    return [[None, welcome_markdown]]

def append_chat_turn(history, user_msg, bot_reply):
    history = list(history or [])
    if CHAT_USE_MESSAGES:
        history.append({"role": "user", "content": user_msg})
        history.append({"role": "assistant", "content": bot_reply})
    else:
        history.append([user_msg, bot_reply])
    return history

def create_chatbot(initial_val, **kwargs):
    if CHAT_SUPPORTS_TYPE:
        return gr.Chatbot(value=initial_val, type="messages", **kwargs)
    return gr.Chatbot(value=initial_val, **kwargs)


# ---------------- Modern AI Studio Theme & CSS ----------------
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --bg-main: #070913;
    --bg-panel: rgba(17, 24, 39, 0.78);
    --bg-card: rgba(30, 41, 59, 0.55);
    --border-glow: rgba(99, 102, 241, 0.28);
    --accent-primary: #6366f1;
    --accent-secondary: #06b6d4;
    --text-primary: #f8fafc;
    --text-muted: #94a3b8;
}

.gradio-container {
    background: radial-gradient(circle at 10% 10%, #131b36 0%, #070913 45%, #05070e 100%) !important;
    font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
    color: var(--text-primary) !important;
    max-width: 1440px !important;
    margin: 0 auto !important;
}

/* Header Banner */
.ai-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 20px 28px;
    border-radius: 18px;
    background: linear-gradient(135deg, rgba(49, 46, 129, 0.75) 0%, rgba(15, 23, 42, 0.9) 60%, rgba(8, 145, 178, 0.45) 100%);
    border: 1px solid rgba(129, 140, 248, 0.35);
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.45), inset 0 1px 0 rgba(255, 255, 255, 0.12);
    backdrop-filter: blur(16px);
    margin-bottom: 16px;
}
.ai-brand { display: flex; align-items: center; gap: 16px; }
.ai-orb {
    width: 46px; height: 46px; border-radius: 14px;
    background: linear-gradient(135deg, #6366f1, #06b6d4);
    display: flex; align-items: center; justify-content: center;
    font-weight: 800; font-size: 20px; color: #fff;
    box-shadow: 0 0 24px rgba(99, 102, 241, 0.65);
}
.ai-title h1 {
    margin: 0; font-size: 24px; font-weight: 700; letter-spacing: -0.02em;
    background: linear-gradient(90deg, #ffffff, #a5f3fc);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.ai-title p { margin: 4px 0 0; font-size: 13.5px; color: #cbd5e1; }
.ai-badge {
    padding: 6px 14px; border-radius: 999px; font-size: 12px; font-weight: 600;
    background: rgba(6, 182, 212, 0.15); border: 1px solid rgba(6, 182, 212, 0.45);
    color: #67e8f9; letter-spacing: 0.03em;
}

/* KPI Metric Cards */
.cards { display: grid; grid-template-columns: repeat(auto-fit, minmax(155px, 1fr)); gap: 12px; margin: 12px 0; }
.card {
    padding: 16px; border-radius: 14px;
    background: linear-gradient(180deg, rgba(30, 41, 59, 0.65) 0%, rgba(15, 23, 42, 0.85) 100%);
    border: 1px solid rgba(148, 163, 184, 0.16);
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.25);
    transition: all 0.2s ease;
}
.card:hover {
    transform: translateY(-3px);
    border-color: rgba(99, 102, 241, 0.55);
    box-shadow: 0 8px 25px rgba(99, 102, 241, 0.2);
}
.card b {
    display: block; font-size: 21px; font-weight: 700;
    color: #38bdf8; margin-bottom: 4px;
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis;
}
.card span { font-size: 12px; font-weight: 500; color: #94a3b8; text-transform: uppercase; letter-spacing: 0.04em; }

/* Audit Log Pill Box */
.audit-box {
    padding: 14px 18px; border-radius: 14px;
    background: rgba(16, 185, 129, 0.08);
    border: 1px solid rgba(16, 185, 129, 0.3);
    color: #d1fae5; font-size: 13.5px; line-height: 1.6;
    margin: 10px 0;
}
.audit-box h4 { margin: 0 0 6px 0; color: #34d399; font-size: 14px; }

.hint { color: #94a3b8; font-size: 13px; }
"""

PLOTLY_LAYOUT = dict(
    paper_bgcolor="rgba(15, 23, 42, 0.0)",
    plot_bgcolor="rgba(15, 23, 42, 0.35)",
    font=dict(family="Plus Jakarta Sans, sans-serif", color="#e2e8f0", size=12),
    title_font=dict(size=16, color="#f8fafc"),
    margin=dict(l=40, r=24, t=52, b=40),
    xaxis=dict(gridcolor="rgba(148, 163, 184, 0.12)", zerolinecolor="rgba(148, 163, 184, 0.2)"),
    yaxis=dict(gridcolor="rgba(148, 163, 184, 0.12)", zerolinecolor="rgba(148, 163, 184, 0.2)"),
    colorway=["#6366f1", "#06b6d4", "#10b981", "#f59e0b", "#ec4899", "#8b5cf6", "#3b82f6"]
)

def style_fig(fig, title=None):
    if fig is None:
        return None
    fig.update_layout(**PLOTLY_LAYOUT)
    if title:
        fig.update_layout(title_text=title)
    return fig


# ---------------- Core Data & Smart Auto-Prep Layer ----------------
def cards(items):
    return "<div class='cards'>" + "".join(
        f"<div class='card' title='{v}'><b>{v}</b><span>{k}</span></div>" for k, v in items
    ) + "</div>"

def read_any(path, sheet=None):
    ext = os.path.splitext(path)[1].lower()
    if ext not in (".csv", ".tsv", ".xlsx", ".xls", ".json"):
        raise ValueError(f"Unsupported file type '{ext}'. Use CSV, TSV, XLSX, XLS or JSON.")
    if os.path.getsize(path) > MAX_MB * 1024 * 1024:
        raise ValueError(f"File is larger than {MAX_MB} MB. Split it or load a subset outside this app.")
    if ext == ".csv":
        try:
            return pd.read_csv(path, sep=None, engine="python")
        except UnicodeDecodeError:
            return pd.read_csv(path, sep=None, engine="python", encoding="latin-1")
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t")
    if ext == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    return pd.read_excel(path, sheet_name=sheet or 0)

def dedupe_columns(df):
    seen, new = {}, []
    for c in df.columns:
        c_clean = re.sub(r"\s+", " ", str(c).strip()) or "Unnamed_Col"
        seen[c_clean] = seen.get(c_clean, 0) + 1
        new.append(c_clean if seen[c_clean] == 1 else f"{c_clean}_{seen[c_clean]}")
    renamed = [n for n, o in zip(new, df.columns) if n != str(o)]
    df.columns = new
    return df, renamed

def sheets_url(url):
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", url or "")
    if not m:
        raise ValueError("That does not look like a Google Sheets URL.")
    g = re.search(r"gid=(\d+)", url)
    return f"https://docs.google.com/spreadsheets/d/{m.group(1)}/export?format=csv" + (f"&gid={g.group(1)}" if g else "")

NULL_TOKENS = {"", "nan", "none", "null", "n/a", "na", "-", "--", "?", "nil", "undefined"}

def miss_mask(df):
    m = df.isna().copy()
    for i, c in enumerate(df.columns):
        if df.dtypes.iloc[i] == object:
            m[c] = m[c] | df[c].astype(str).str.strip().str.lower().isin(NULL_TOKENS)
    return m

def col_kind(s, name):
    n = s.dropna()
    if n.empty:
        return "empty"
    if pd.api.types.is_bool_dtype(s):
        return "bool"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "date"
    if pd.api.types.is_numeric_dtype(s):
        return "id" if ID_RE.search(str(name)) and n.nunique() > 0.9 * len(n) else "num"
    t = n.astype(str).str.strip()
    t = t[~t.str.lower().isin(NULL_TOKENS)]
    if t.empty:
        return "empty"
    if t.str.lower().isin(["true", "false", "yes", "no"]).all():
        return "bool"
    cleaned_num = t.str.replace(r"[$€£₹,%]", "", regex=True).str.replace(",", "", regex=False).str.strip()
    if pd.to_numeric(cleaned_num, errors="coerce").notna().mean() >= 0.92:
        return "num"
    if t.str.contains(r"\d").all() and pd.to_datetime(t.head(500), errors="coerce").notna().mean() >= 0.88:
        return "date"
    if ID_RE.search(str(name)) and t.nunique() > 0.9 * len(t):
        return "id"
    return "text" if t.nunique() > 50 and t.nunique() > 0.5 * len(t) else "cat"

def typed_frame(df):
    out, kinds = df.copy(), {}
    for c in df.columns:
        k = col_kind(df[c], c)
        kinds[c] = k
        if df[c].dtype == object:
            t = df[c].astype(str).str.strip()
            t = t.where(~t.str.lower().isin(NULL_TOKENS), np.nan)
            if k in ("num", "id"):
                clean_s = t.str.replace(r"[$€£₹,%]", "", regex=True).str.replace(",", "", regex=False).str.strip()
                if k == "num" or pd.to_numeric(clean_s.dropna(), errors="coerce").notna().all():
                    out[c] = pd.to_numeric(clean_s, errors="coerce")
                else:
                    out[c] = t
            elif k == "date":
                out[c] = pd.to_datetime(t, errors="coerce")
            elif k == "bool":
                out[c] = t.str.lower().map({"true": True, "false": False, "yes": True, "no": False})
            else:
                out[c] = t
    return out, kinds

def completeness(df):
    cells = df.shape[0] * df.shape[1]
    return 100.0 if cells == 0 else 100.0 * (1.0 - miss_mask(df).values.sum() / cells)

def save_ready_excel(df, base_name="datapilot_ready"):
    clean_name = re.sub(r"[^a-zA-Z0-9_-]", "_", os.path.splitext(base_name)[0])
    path = os.path.join(tempfile.gettempdir(), f"{clean_name}_Analysis_Ready.xlsx")
    d2 = df.copy()
    for c in d2.columns:
        if isinstance(d2[c].dtype, pd.DatetimeTZDtype):
            d2[c] = d2[c].dt.tz_localize(None)
    d2.to_excel(path, index=False)
    return path

def auto_prep_pipeline(df):
    """
    Automatically removes unexpected artifacts, drops exact duplicates,
    parses numbers/dates, and imputes missing values so the dataset is immediately ready for analysis.
    """
    out = df.copy()
    logs = []
    init_rows, init_cols = out.shape

    # 1. Strip control characters, currency noise, & standardize null tokens
    obj_cols = out.select_dtypes(include=["object"]).columns
    unexpected_fixed = 0
    for c in obj_cols:
        raw_s = out[c].astype(str)
        cleaned = (
            raw_s.str.replace(r"[\x00-\x1f\x7f-\x9f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        is_null = out[c].isna() | cleaned.str.lower().isin(NULL_TOKENS)
        unexpected_fixed += int((raw_s != cleaned).sum())
        out[c] = cleaned.where(~is_null, np.nan)
    if unexpected_fixed > 0:
        logs.append(f"Sanitized <b>{unexpected_fixed:,}</b> cells containing extra whitespace, hidden characters, or placeholder null strings.")

    # 2. Drop completely empty rows and columns
    empty_cols = [c for c in out.columns if out[c].isna().all()]
    if empty_cols:
        out = out.drop(columns=empty_cols)
        logs.append(f"Removed <b>{len(empty_cols)}</b> completely empty column(s): {', '.join(empty_cols[:5])}.")
    before_empty_rows = len(out)
    out = out.dropna(how="all").reset_index(drop=True)
    if len(out) < before_empty_rows:
        logs.append(f"Removed <b>{before_empty_rows - len(out):,}</b> completely blank row(s).")

    # 3. Drop exact duplicate rows
    dup_count = int(out.duplicated().sum())
    if dup_count > 0:
        out = out.drop_duplicates().reset_index(drop=True)
        logs.append(f"Removed <b>{dup_count:,}</b> exact duplicate row(s).")

    # 4. Type inference & conversion (currency/commas -> numeric, date strings -> datetime)
    typed_df, kinds = typed_frame(out)
    out = typed_df

    # 5. Smart Missing Value Imputation (Median for numeric, Mode/'Unknown' for categorical)
    num_imputed, cat_imputed = 0, 0
    for c in out.columns:
        k = kinds.get(c)
        na_cnt = int(out[c].isna().sum())
        if na_cnt == 0:
            continue
        if k == "num":
            med = out[c].median()
            fill_val = med if pd.notna(med) else 0
            out[c] = out[c].fillna(fill_val)
            num_imputed += na_cnt
        elif k in ("cat", "text"):
            modes = out[c].mode(dropna=True)
            fill_val = modes.iloc[0] if not modes.empty else "Unknown"
            out[c] = out[c].fillna(fill_val)
            cat_imputed += na_cnt
        elif k == "bool":
            modes = out[c].mode(dropna=True)
            fill_val = bool(modes.iloc[0]) if not modes.empty else False
            out[c] = out[c].fillna(fill_val)
            cat_imputed += na_cnt

    if num_imputed > 0:
        logs.append(f"Imputed <b>{num_imputed:,}</b> missing numerical values using column medians.")
    if cat_imputed > 0:
        logs.append(f"Filled <b>{cat_imputed:,}</b> missing categorical/text values using most frequent values.")

    if not logs:
        logs.append("Dataset was already clean! No duplicate rows, missing cells, or formatting anomalies found.")

    summary_html = (
        "<div class='audit-box'><h4>✨ Smart Auto-Prep Pipeline Complete</h4>"
        f"<ul style='margin:0;padding-left:18px'>{''.join(f'<li>{item}</li>' for item in logs)}</ul>"
        f"<div style='margin-top:6px;font-size:12px;color:#a7f3d0'>Shape: {init_rows:,} × {init_cols} → <b>{len(out):,} × {out.shape[1]}</b> | Completeness: <b>{completeness(out):.1f}%</b></div>"
        "</div>"
    )
    return out, logs, summary_html


def refresh(st):
    st["typed"], st["kinds"] = typed_frame(st["work"])
    st["fidx"], st["filters"] = st["typed"].index, []
    st["ready_path"] = save_ready_excel(st["work"], st.get("name", "dataset"))
    return st

def cur(st):
    return st["typed"].loc[st["fidx"]]

def need(st):
    if not st or "orig" not in st:
        raise gr.Error("Please upload a dataset first in the AI Workspace & Upload tab.")
    return st

def of_kind(st, *ks):
    return [c for c in st["typed"].columns if st["kinds"][c] in ks]


# ---------------- Upload & Overview ----------------
def on_file(f):
    if f is None:
        return gr.update(choices=[], value=None)
    if str(f).lower().endswith((".xlsx", ".xls")):
        try:
            names = pd.ExcelFile(f).sheet_names
            return gr.update(choices=names, value=names[0])
        except Exception as e:
            raise gr.Error(f"Could not read the workbook: {e}")
    return gr.update(choices=[], value=None)

def dd(cols):
    cols = list(cols)
    return [
        gr.update(choices=cols, value=None),
        gr.update(choices=cols, value=[]),
        gr.update(choices=cols, value=None),
        gr.update(choices=cols, value=cols[0] if cols else None),
        gr.update(choices=cols, value=None),
        gr.update(choices=["(none)"] + cols, value="(none)"),
        gr.update(choices=["(none)"] + cols, value="(none)"),
    ]

def overview(st):
    df, k = st["work"], st["kinds"]
    cnt = lambda *ks: sum(1 for v in k.values() if v in ks)
    return cards([
        ("Dataset", st["name"]),
        ("Format", st["fmt"]),
        ("Active Rows", f"{len(df):,}"),
        ("Columns", df.shape[1]),
        ("Numerical", cnt("num")),
        ("Categorical", cnt("cat", "text")),
        ("Date/Time", cnt("date")),
        ("Missing Cells", int(miss_mask(df).values.sum())),
        ("Duplicates", int(df.duplicated().sum())),
        ("Completeness", f"{completeness(df):.1f}%"),
    ])

def profile_table(st):
    df, t, k = st["work"], st["typed"], st["kinds"]
    m = miss_mask(df)
    rows = []
    for i, c in enumerate(df.columns):
        s = t[c]
        r = {
            "#": i + 1,
            "Column": c,
            "Detected Type": k[c].upper(),
            "Dtype": str(df[c].dtype),
            "Unique": int(s.nunique()),
            "Missing": int(m[c].sum()),
            "Missing %": round(100.0 * m[c].mean(), 2) if len(df) else 0,
        }
        if k[c] in ("num", "id") and s.notna().any():
            r.update(Min=round(float(s.min()), 3), Max=round(float(s.max()), 3), Mean=round(float(s.mean()), 3), Median=round(float(s.median()), 3))
        elif k[c] == "date" and s.notna().any():
            r.update(Min=str(s.min().date()), Max=str(s.max().date()))
        elif k[c] in ("cat", "text", "bool") and s.notna().any():
            r["Top Value"] = str(s.value_counts().index[0])
        rows.append(r)
    return pd.DataFrame(rows)

def preview(n, st):
    return need(st)["work"].head(int(n))

def load_data(file, sheet, url, auto_clean, n, st):
    try:
        if url and url.strip():
            raw_df = pd.read_csv(sheets_url(url.strip()))
            name, fmt, sh, size = "Google_Sheet", "GSHEET", None, "Cloud"
        elif file is not None:
            raw_df = read_any(file, sheet)
            name = os.path.basename(file)
            fmt = os.path.splitext(file)[1].upper().strip(".")
            sh = sheet
            size = f"{os.path.getsize(file) / 1024:.1f} KB"
        else:
            raise ValueError("Upload a file (CSV/Excel/JSON) or paste a public Google Sheets link first.")
    except Exception as e:
        msg = f"❌ **Could not load data:** {e}"
        return [st, msg, "", "", None, None, None, []] + [gr.update()] * 7

    if raw_df.empty and raw_df.shape[1] == 0:
        return [st, "❌ **Error:** The file contains no columns.", "", "", None, None, None, []] + [gr.update()] * 7

    raw_df, renamed = dedupe_columns(raw_df)
    if auto_clean:
        work_df, logs, audit_html = auto_prep_pipeline(raw_df)
    else:
        work_df = raw_df.copy()
        logs = ["Auto-clean bypassed by user. Loaded raw data as-is."]
        audit_html = "<div class='audit-box'><h4>ℹ️ Raw Mode</h4>Loaded raw dataset without automatic imputation or deduplication.</div>"

    st = refresh({
        "orig": raw_df,
        "work": work_df,
        "name": name,
        "fmt": fmt,
        "sheet": sh,
        "size": size,
        "logs": logs,
        "audit_html": audit_html,
    })

    nums = of_kind(st, "num")
    cats = of_kind(st, "cat", "text")
    welcome_bot = (
        f"### ✅ **{name}** is loaded and ready for analysis!\n"
        f"- **Dimensions:** `{len(work_df):,}` rows × `{work_df.shape[1]}` columns (`{completeness(work_df):.1f}%` complete)\n"
        f"- **Detected Schema:** `{len(nums)}` Numerical, `{len(cats)}` Categorical, `{len(of_kind(st, 'date'))}` Date/Time\n"
        f"- **Auto-Cleaning Summary:** {' '.join(s.replace('<b>', '**').replace('</b>', '**') for s in logs)}\n\n"
        f"💡 *Try asking me:* **\"Give me an executive summary\"**, **\"Plot {nums[0] if nums else 'counts'} by {cats[0] if cats else 'category'}\"**, **\"Show outliers\"**, or **\"Show cleaning report\"**."
    )
    chat_init = format_chat_init(welcome_bot)
    note = f" *(Renamed duplicate headers: {', '.join(renamed)})*" if renamed else ""
    status_md = f"✅ **Loaded `{name}`** — `{len(work_df):,}` rows × `{work_df.shape[1]}` cols.{note}"

    return [
        st,
        status_md,
        audit_html,
        overview(st),
        profile_table(st),
        work_df.head(int(n)),
        st["ready_path"],
        chat_init,
    ] + dd(work_df.columns)


# ---------------- Conversational AI Copilot Engine ----------------
def ai_copilot_reply(message, history, st):
    history = list(history or [])
    if not message or not message.strip():
        return history, None, None

    user_msg = message.strip()

    if not st or "work" not in st:
        bot_warn = "⚠️ Please **upload an Excel/CSV dataset** on the left panel first so I can analyze your numbers, generate charts, and prepare your clean workbook."
        return append_chat_turn(history, user_msg, bot_warn), None, None

    d = cur(st)
    k = st["kinds"]
    nums = of_kind(st, "num")
    cats = of_kind(st, "cat", "text", "bool")
    dates = of_kind(st, "date")
    q = user_msg.lower()

    fig, out_df = None, None

    matched_cols = [c for c in d.columns if c.lower() in q]
    matched_nums = [c for c in matched_cols if k[c] == "num"] or nums
    matched_cats = [c for c in matched_cols if k[c] in ("cat", "text", "bool")] or cats

    # 1. Cleaning / Audit / Ready Excel queries
    if any(w in q for w in ["clean", "duplicate", "missing", "audit", "ready", "excel", "fix"]):
        m_cnt = int(miss_mask(d).values.sum())
        dup_cnt = int(d.duplicated().sum())
        reply = (
            f"### 🧹 Smart Data Hygiene Report (`{st['name']}`)\n"
            f"- **Original Rows:** `{len(st['orig']):,}` → **Cleaned Working Rows:** `{len(st['work']):,}`\n"
            f"- **Remaining Missing Cells:** `{m_cnt:,}` | **Remaining Duplicates:** `{dup_cnt:,}`\n"
            f"- **Completeness Score:** `{completeness(d):.2f}%`\n\n"
            "**Automated Pipeline Actions Taken:**\n" +
            "\n".join(f"- {item.replace('<b>', '**').replace('</b>', '**')}" for item in st.get("logs", [])) +
            "\n\n📥 *Your cleaned, analysis-ready `.xlsx` file is available for immediate download in the left sidebar!*"
        )
        out_df = missing_table(st)

    # 2. Outlier Detection
    elif any(w in q for w in ["outlier", "anomaly", "anomalies", "extreme", "iqr"]):
        _, _, out_df, _, _ = eda(st)
        if nums:
            target = matched_nums[0]
            fig = make_chart(d, k, "Box plot", matched_cats[0] if matched_cats else None, target, None, "Count", 10)[0]
            reply = (
                f"### 🔍 IQR Outlier Inspection\n"
                f"I scanned `{len(nums)}` numerical column(s) using the `1.5 × IQR` rule. "
                f"Below is the outlier boundary table and the box plot distribution for **`{target}`**."
            )
        else:
            reply = "No numerical columns were found to test for outliers."

    # 3. Correlation & Relationships
    elif any(w in q for w in ["corr", "relationship", "heatmap", "versus", " vs "]):
        if len(nums) >= 2:
            c_matrix = d[nums].corr()
            fig = make_chart(d, k, "Correlation heatmap", None, None, None, "Count", 10)[0]
            pairs = (
                c_matrix.where(np.triu(np.ones(c_matrix.shape), k=1).astype(bool))
                .stack()
                .reset_index()
            )
            pairs.columns = ["Column A", "Column B", "Correlation (r)"]
            pairs["Abs"] = pairs["Correlation (r)"].abs()
            out_df = pairs.sort_values("Abs", ascending=False).drop(columns="Abs").head(10).round(3)
            top_row = out_df.iloc[0] if not out_df.empty else None
            top_txt = f"The strongest relationship is between **`{top_row['Column A']}`** and **`{top_row['Column B']}`** (`r = {top_row['Correlation (r)']:.2f}`)." if top_row is not None else ""
            reply = f"### 🔗 Correlation Matrix Analysis\n{top_txt}\nSee the interactive heatmap and top correlated pairs below."
        else:
            reply = "At least **2 numerical columns** are required to compute a correlation matrix."

    # 4. Trend / Time-series
    elif any(w in q for w in ["trend", "time", "month", "year", "date", "over time"]) and dates:
        dc = dates[0]
        yc = matched_nums[0] if matched_nums else None
        tmp = d.dropna(subset=[dc]).copy()
        tmp["Period"] = tmp[dc].dt.to_period("M").dt.to_timestamp()
        if yc:
            g = tmp.groupby("Period")[yc].sum().reset_index()
            fig = style_fig(px.area(g, x="Period", y=yc, markers=True), f"Monthly {yc} Trend ({dc})")
            reply = f"### 📈 Time-Series Trend: `{yc}` over `{dc}`\nAggregated monthly totals across `{len(g)}` active periods."
            out_df = g.tail(12)
        else:
            g = tmp.groupby("Period").size().reset_index(name="Record Count")
            fig = style_fig(px.line(g, x="Period", y="Record Count", markers=True), f"Monthly Volume ({dc})")
            reply = f"### 📈 Record Volume Over Time (`{dc}`)"
            out_df = g.tail(12)

    # 5. Chart / Plot / Distribution requests
    elif any(w in q for w in ["chart", "plot", "graph", "visual", "bar", "pie", "scatter", "hist", "distribution", "top"]):
        if "scatter" in q and len(matched_nums) >= 2:
            fig, _ = make_chart(d, k, "Scatter plot", matched_nums[0], matched_nums[1], matched_cats[0] if matched_cats else None, "Count", 12)
            reply = f"### 📊 Scatter Plot: `{matched_nums[0]}` vs `{matched_nums[1]}`"
        elif "pie" in q or "donut" in q:
            cx = matched_cats[0] if matched_cats else d.columns[0]
            cy = matched_nums[0] if matched_nums else None
            fig, _ = make_chart(d, k, "Donut chart", cx, cy, None, "Sum" if cy else "Count", 10)
            reply = f"### 🍩 Share Breakdown by `{cx}`"
        elif "hist" in q or ("distribution" in q and matched_nums):
            cx = matched_nums[0]
            fig, _ = make_chart(d, k, "Histogram", cx, None, matched_cats[0] if matched_cats else None, "Count", 15)
            reply = f"### 📊 Distribution of `{cx}`"
        else:
            cx = matched_cats[0] if matched_cats else (matched_nums[0] if matched_nums else d.columns[0])
            cy = matched_nums[0] if matched_nums and cx not in matched_nums else None
            agg_mode = "Sum" if cy else "Count"
            fig, _ = make_chart(d, k, "Bar chart", cx, cy, None, agg_mode, 12)
            reply = f"### 📊 Top Categories in `{cx}` ({agg_mode} {cy or 'Rows'})"
        out_df = d.head(15)

    # 6. Default: Comprehensive Executive Summary & KPI Breakdown
    else:
        num_lines = []
        for c in nums[:5]:
            s = d[c].dropna()
            if not s.empty:
                num_lines.append(f"- **`{c}`**: Sum = `{s.sum():,.2f}` | Mean = `{s.mean():,.2f}` | Median = `{s.median():,.2f}` | Range = `[{s.min():,.2f} → {s.max():,.2f}]`")
        cat_lines = []
        for c in cats[:4]:
            vc = d[c].value_counts(dropna=True)
            if not vc.empty:
                cat_lines.append(f"- **`{c}`** (`{vc.size}` unique): Top is **`{vc.index[0]}`** (`{vc.iloc[0]:,}` records, `{100 * vc.iloc[0] / len(d):.1f}%`)")

        reply = (
            f"### 🧠 Executive Insights for `{st['name']}`\n"
            f"**Active Scope:** `{len(d):,}` rows × `{d.shape[1]}` columns | **Data Completeness:** `{completeness(d):.1f}%`\n\n"
            + ("**Key Numerical Metrics:**\n" + "\n".join(num_lines) + "\n\n" if num_lines else "")
            + ("**Dominant Categories:**\n" + "\n".join(cat_lines) if cat_lines else "")
        )
        out_df = profile_table(st)
        if cats and nums:
            fig, _ = make_chart(d, k, "Bar chart", cats[0], nums[0], None, "Sum", 10)
        elif nums:
            fig, _ = make_chart(d, k, "Histogram", nums[0], None, None, "Count", 10)

    return append_chat_turn(history, user_msg, reply), fig, out_df


# ---------------- Data Quality Inspector ----------------
def missing_table(st):
    df = need(st)["work"]
    m = miss_mask(df)
    return pd.DataFrame({
        "Column": df.columns,
        "Position": range(1, df.shape[1] + 1),
        "Missing": m.sum().values,
        "Missing %": (100.0 * m.mean()).round(2).values if len(df) else 0,
        "Non-missing": (~m).sum().values,
        "Data type": [str(t) for t in df.dtypes],
    })

def missing_rows(col, st):
    df = need(st)["work"]
    m = miss_mask(df)
    sel = m[col] if col else m.any(axis=1)
    out = df[sel].copy()
    out.insert(0, "Row #", out.index + 1)
    return out.head(1000)

def dup_view(mode, cols, st):
    df = need(st)["work"]
    cols = list(cols or [])
    if mode == "full":
        out = df[df.duplicated(keep=False)]
    elif mode == "unique":
        out = df.drop_duplicates()
    elif mode == "potential":
        if not cols:
            raise gr.Error("Select at least one column under 'Columns to compare'.")
        out = df[df.duplicated(subset=cols, keep=False)].sort_values(cols)
    else:
        if not cols:
            raise gr.Error("Select a column under 'Columns to compare'.")
        c = cols[0]
        vc = df[c].value_counts(dropna=True)
        vc = vc[vc > 1]
        return pd.DataFrame({
            c: vc.index,
            "Count": vc.values,
            "% of rows": (100.0 * vc.values / len(df)).round(2),
        }).head(500)
    out = out.copy()
    out.insert(0, "Row #", out.index + 1)
    return out.head(1000)

def type_issues(st):
    df, t, k = need(st)["work"], st["typed"], st["kinds"]
    rows = []
    for c in df.columns:
        n = df[c].dropna()
        if n.empty:
            rows.append((c, "All values missing", "Consider removing the column"))
            continue
        if df[c].dtype == object:
            if k[c] == "num":
                rows.append((c, "Numbers stored as text", "Auto-converted in typed layer or use Cleaning tab"))
            if k[c] == "date":
                rows.append((c, "Dates stored as text", "Convert to datetime (Cleaning tab)"))
            if n.map(type).nunique() > 1:
                rows.append((c, "Mixed Python types", "Convert to text or clean values"))
            if k[c] in ("cat",):
                s = n.astype(str)
                canon = s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
                if canon.nunique() < s.nunique():
                    rows.append((c, "Inconsistent labels (case/spacing variants)", "Standardize labels (Cleaning tab)"))
        if k[c] == "date" and df[c].dtype == object:
            bad = int(t[c].isna().sum() - miss_mask(df)[c].sum())
            if bad > 0:
                rows.append((c, f"{bad} unparseable date strings", "Inspect rows then fix"))
    return pd.DataFrame(rows, columns=["Column", "Possible issue", "Suggestion"]) if rows else pd.DataFrame({"Result": ["✅ No type issues detected — dataset is clean!"]})

def quality_summary(st):
    df = need(st)["work"]
    m = miss_mask(df)
    rep = int(sum(1 for c in df.columns if df[c].duplicated().any()))
    iss = type_issues(st)
    n_iss = 0 if "Result" in iss.columns else len(iss)
    return cards([
        ("Missing cells", f"{int(m.values.sum()):,}"),
        ("Duplicate rows", int(df.duplicated().sum())),
        ("Columns with missing", int(m.any().sum())),
        ("Cols w/ repeated vals", rep),
        ("Type issues", n_iss),
        ("Completeness", f"{completeness(df):.2f}%"),
    ])


# ---------------- Universal Data Explorer (Filters) ----------------
def apply_one(s, kind, op, v1, v2):
    v1, v2 = (v1 or "").strip(), (v2 or "").strip()
    if op == "is missing":
        return s.isna()
    if op == "not missing":
        return s.notna()
    if kind == "num" or (kind == "id" and pd.api.types.is_numeric_dtype(s)):
        a = float(v1) if v1 else None
        b = float(v2) if v2 else None
        if op == "between":
            return (s >= (a if a is not None else -np.inf)) & (s <= (b if b is not None else np.inf))
        if op == ">=":
            return s >= a
        if op == "<=":
            return s <= a
        if op == "equals":
            return s == a
    elif kind == "date":
        a = pd.to_datetime(v1) if v1 else pd.Timestamp.min
        b = pd.to_datetime(v2) if v2 else pd.Timestamp.max
        if op == "between":
            return (s >= a) & (s <= b)
    elif kind == "bool":
        if op == "equals":
            return s == (v1.lower() == "true")
    else:
        t = s.astype(str).str.lower()
        q = v1.lower()
        if op == "in list":
            return t.isin([x.strip().lower() for x in v1.split(",")]) & s.notna()
        if op == "equals":
            return (t == q) & s.notna()
        if op == "contains":
            return t.str.contains(q, regex=False) & s.notna()
        if op == "starts with":
            return t.str.startswith(q) & s.notna()
        if op == "ends with":
            return t.str.endswith(q) & s.notna()
    raise ValueError(f"Operator '{op}' is not valid for a {kind} column.")

OPS = {
    "num": ["between", ">=", "<=", "equals", "is missing", "not missing"],
    "date": ["between", "is missing", "not missing"],
    "bool": ["equals", "is missing", "not missing"],
}
TXT_OPS = ["in list", "equals", "contains", "starts with", "ends with", "is missing", "not missing"]

def col_changed(col, st):
    if not st or "kinds" not in st or not col:
        return gr.update(), ""
    k = st["kinds"][col]
    s = st["typed"][col]
    if k == "num":
        h = f"Numeric range: `{s.min():g}` to `{s.max():g}`. Use `between` with min and max."
    elif k == "date":
        h = f"Dates: `{s.min()}` to `{s.max()}`. Enter `YYYY-MM-DD`."
    elif k == "bool":
        h = "Enter `True` or `False`."
    elif k == "id":
        h = "Identifier column: use `equals` or `in list` (comma-separated)."
    else:
        top_vals = ", ".join(map(str, s.dropna().astype(str).value_counts().head(10).index))
        h = f"Sample values: {top_vals}" + (" ..." if s.nunique() > 10 else "")
    ops = OPS.get(k, TXT_OPS)
    return gr.update(choices=ops, value=ops[0]), h

def filter_summary(st):
    if not st["filters"]:
        return f"No filters active. Showing all {len(st['typed']):,} rows."
    return (
        "Active filters (all must match):\n"
        + "\n".join(f"- {c} {o} {a} {b}".strip() for c, o, a, b in st["filters"])
        + f"\n\nMatching rows: {len(st['fidx']):,} of {len(st['typed']):,}"
    )

def run_filters(st):
    t = st["typed"]
    mask = pd.Series(True, index=t.index)
    for c, o, a, b in st["filters"]:
        mask &= apply_one(t[c], st["kinds"][c], o, a, b).fillna(False)
    st["fidx"] = t.index[mask]
    return st

def filter_outputs(st):
    d = cur(st)
    num = d[of_kind(st, "num")]
    stats = (
        num.describe().T.round(3).reset_index().rename(columns={"index": "Column"})
        if num.shape[1]
        else pd.DataFrame({"Info": ["No numerical columns"]})
    )
    return st, filter_summary(st), d.head(100), stats

def add_filter(col, op, v1, v2, st):
    need(st)
    if not col or col not in st["typed"].columns:
        raise gr.Error("Choose a column to filter.")
    try:
        apply_one(st["typed"][col].head(5), st["kinds"][col], op, v1, v2)
    except Exception as e:
        raise gr.Error(f"Invalid filter: {e}")
    st["filters"].append((col, op, v1 or "", v2 or ""))
    return filter_outputs(run_filters(st))

def undo_filter(st):
    need(st)
    if st["filters"]:
        st["filters"].pop()
    return filter_outputs(run_filters(st))

def reset_filters(st):
    need(st)
    st["filters"] = []
    return filter_outputs(run_filters(st))


# ---------------- Exploratory Data Analysis (EDA) ----------------
def eda(st):
    need(st)
    d = cur(st)
    nums = of_kind(st, "num")
    cats = of_kind(st, "cat", "bool")
    dates = of_kind(st, "date")
    if d.empty:
        raise gr.Error("The filtered dataset has no rows.")

    num_t = pd.DataFrame()
    if nums:
        num_t = d[nums].agg(["count", "mean", "median", "std", "min", "max"]).T
        num_t["Q1"], num_t["Q3"] = d[nums].quantile(0.25), d[nums].quantile(0.75)
        num_t["IQR"] = num_t["Q3"] - num_t["Q1"]
        num_t = num_t.round(3).reset_index().rename(columns={"index": "Column"})

    cat_rows = []
    for c in cats:
        vc = d[c].value_counts()
        cat_rows.append({
            "Column": c,
            "Unique": len(vc),
            "Most frequent": str(vc.index[0]) if len(vc) else "-",
            "Top count": int(vc.iloc[0]) if len(vc) else 0,
            "Top %": round(100.0 * vc.iloc[0] / max(d[c].notna().sum(), 1), 1) if len(vc) else 0,
            "Missing": int(d[c].isna().sum()),
        })

    out_rows = []
    for c in nums:
        s = d[c].dropna()
        if len(s) < 4:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        out_rows.append({
            "Column": c,
            "Lower bound": round(float(lo), 3),
            "Upper bound": round(float(hi), 3),
            "Potential outliers": n_out,
            "% flagged": round(100.0 * n_out / len(s), 2),
        })

    date_rows = [{
        "Column": c,
        "Earliest": str(d[c].min()),
        "Latest": str(d[c].max()),
        "Span (days)": (d[c].max() - d[c].min()).days if d[c].notna().any() else None,
        "Missing": int(d[c].isna().sum()),
    } for c in dates]

    empty = lambda m: pd.DataFrame({"Info": [m]})
    corr = None
    if len(nums) >= 2:
        corr = style_fig(
            px.imshow(
                d[nums].corr(),
                text_auto=".2f",
                color_continuous_scale="RdBu_r",
                zmin=-1,
                zmax=1,
            ),
            "Correlation Matrix (Numerical Columns)"
        )
    return (
        num_t if nums else empty("No numerical columns"),
        pd.DataFrame(cat_rows) if cat_rows else empty("No categorical columns"),
        pd.DataFrame(out_rows) if out_rows else empty("No outlier analysis possible"),
        pd.DataFrame(date_rows) if date_rows else empty("No date columns"),
        corr,
    )


# ---------------- Visual Analytics Studio ----------------
AGGS = {"Count": "size", "Sum": "sum", "Mean": "mean", "Median": "median", "Minimum": "min", "Maximum": "max"}
CHARTS = ["Bar chart", "Line chart", "Area chart", "Histogram", "Box plot", "Scatter plot", "Pie chart", "Donut chart", "Correlation heatmap"]

def make_chart(d, kinds, ctype, x, y, color, agg, topn):
    color = None if color in (None, "", "(none)") else color
    y = None if y in (None, "", "(none)") else y
    num = lambda c: c and kinds.get(c) in ("num", "id")

    if d.empty:
        return None, "No rows to plot (check your active filters)."
    if ctype == "Correlation heatmap":
        nums = [c for c in d.columns if kinds[c] == "num"]
        if len(nums) < 2:
            return None, "A correlation heatmap needs at least two numerical columns."
        fig = px.imshow(d[nums].corr(), text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r")
        return style_fig(fig, "Correlation Matrix"), ""

    if not x and ctype != "Box plot":
        return None, "Choose an X-axis column."
    if ctype == "Histogram":
        if not num(x):
            return None, "A histogram needs a numerical X column."
        return style_fig(px.histogram(d, x=x, color=color), f"Distribution of {x}"), ""
    if ctype == "Box plot":
        yy = y or (x if num(x) else None)
        if not num(yy):
            return None, "A box plot needs a numerical Y column."
        return style_fig(px.box(d, x=None if yy == x else x, y=yy, color=color), f"Box Plot of {yy}"), ""
    if ctype == "Scatter plot":
        if not (num(x) and num(y)):
            return None, "A scatter plot needs numerical X and Y columns."
        return style_fig(px.scatter(d.head(20000), x=x, y=y, color=color, opacity=0.8), f"{y} vs {x}"), ""

    keys = [x] + ([color] if color and color != x else [])
    if agg != "Count" and not (y and num(y)):
        return None, f"'{agg}' requires a numerical Y column. Pick 'Count' or choose a numerical Y."

    g = (
        d.groupby(keys, dropna=True).size().reset_index(name="Count")
        if agg == "Count" or not y
        else d.groupby(keys, dropna=True)[y].agg(AGGS[agg]).reset_index()
    )
    val = g.columns[-1]
    if g.empty:
        return None, "No data left to plot after grouping."

    is_time = kinds.get(x) == "date"
    if ctype in ("Line chart", "Area chart") or is_time:
        g = g.sort_values(x)
        f = px.area if ctype == "Area chart" else px.line
        return style_fig(f(g, x=x, y=val, color=color if color != x else None, markers=True), f"{agg} of {val} by {x}"), ""

    top = g.groupby(x)[val].sum().nlargest(int(topn)).index
    g = g[g[x].isin(top)].sort_values(val, ascending=False)
    if ctype in ("Pie chart", "Donut chart"):
        fig = px.pie(g, names=x, values=val, hole=0.48 if ctype == "Donut chart" else 0)
        return style_fig(fig, f"{val} Breakdown by {x}"), ""

    return style_fig(px.bar(g, x=x, y=val, color=color if color != x else None, text_auto=".2s"), f"Top {int(topn)} {x} by {val}"), ""

def chart_cb(ctype, x, y, color, agg, topn, st):
    need(st)
    fig, msg = make_chart(cur(st), st["kinds"], ctype, x, y, color, agg, topn)
    return fig, msg or "✅ Chart built from active dataset."

def agg_options(y, st):
    ok = st and "kinds" in st and y and st["kinds"].get(y) == "num"
    ch = list(AGGS) if ok else ["Count"]
    return gr.update(choices=ch, value="Sum" if ok else "Count")


# ---------------- Executive Dashboard Studio ----------------
def dashboard(st):
    need(st)
    d = cur(st)
    k = st["kinds"]
    if d.empty:
        raise gr.Error("No rows match the current filters.")

    nums, cats, dates = of_kind(st, "num"), of_kind(st, "cat", "bool"), of_kind(st, "date")
    kp = [
        ("Active Records", f"{len(d):,}"),
        ("Features", d.shape[1]),
        ("Data Health", f"{completeness(st['work'].loc[st['fidx']]):.1f}%"),
        ("Duplicate Rows", int(st["work"].loc[st["fidx"]].duplicated().sum())),
    ]
    for c in nums[:3]:
        kp.append((f"Avg {c}", f"{d[c].mean():,.2f}"))
        kp.append((f"Sum {c}", f"{d[c].sum():,.2f}"))
    for c in cats[:2]:
        kp.append((f"Distinct {c}", int(d[c].nunique())))
    for c in dates[:1]:
        if d[c].notna().any():
            kp.append((f"{c} Span", f"{d[c].min():%Y-%m-%d} → {d[c].max():%Y-%m-%d}"))

    title = os.path.splitext(st["name"])[0]
    figs = []

    if cats and nums:
        figs.append(make_chart(d, k, "Bar chart", cats[0], nums[0], None, "Sum", 10)[0])
    elif cats:
        figs.append(make_chart(d, k, "Bar chart", cats[0], None, None, "Count", 10)[0])

    if len(cats) >= 2:
        figs.append(make_chart(d, k, "Donut chart", cats[1], nums[0] if nums else None, None, "Sum" if nums else "Count", 8)[0])
    elif cats:
        figs.append(make_chart(d, k, "Donut chart", cats[0], None, None, "Count", 8)[0])

    for c in nums[:2]:
        figs.append(make_chart(d, k, "Histogram", c, None, None, "Count", 12)[0])

    if dates:
        dc = dates[0]
        tmp = d.dropna(subset=[dc]).copy()
        if len(tmp):
            tmp["Period"] = tmp[dc].dt.to_period("M").dt.to_timestamp()
            g = tmp.groupby("Period").size().reset_index(name="Records")
            figs.append(style_fig(px.area(g, x="Period", y="Records", markers=True), f"Monthly Activity ({dc})"))

    if len(nums) >= 2:
        figs.append(make_chart(d, k, "Correlation heatmap", None, None, None, "Count", 10)[0])

    figs = [f for f in figs if f is not None][:6]
    for i, f in enumerate(figs):
        f.update_layout(title_text=f.layout.title.text or f"Executive Insight {i + 1}")

    html = (
        f"<div style='padding:14px 18px;border-radius:14px;background:rgba(30,41,59,0.5);border:1px solid rgba(99,102,241,0.3)'>"
        f"<h2 style='margin:0 0 8px 0;color:#fff'>📊 {title} — Executive Analytics Dashboard</h2>"
        + cards(kp)
        + st.get("audit_html", "")
        + "</div>"
    )

    path = os.path.join(tempfile.gettempdir(), f"{title}_Executive_Dashboard.html")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(
            "<html><head><meta charset='utf-8'><title>DataPilot AI Dashboard</title></head>"
            "<body style='font-family:Inter,sans-serif;background:#070913;color:#f8fafc;padding:28px;max-width:1360px;margin:auto'>"
            + f"<style>{CSS}</style>"
            + html
            + "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(520px,1fr));gap:18px;margin-top:20px'>"
            + "".join(f"<div style='background:#0f172a;padding:12px;border-radius:14px;border:1px solid #1e293b'>{f.to_html(full_html=False, include_plotlyjs='cdn')}</div>" for f in figs)
            + "</div></body></html>"
        )

    plots = figs + [None] * (6 - len(figs))
    return [html, d.head(25), path] + plots


# ---------------- Manual Cleaning & Multi-Format Export ----------------
CLEAN_OPS = [
    "Run Full Auto-Clean Pipeline (dedupe, impute missing, fix text & types)",
    "Remove fully duplicated rows",
    "Fill missing numerical values (median)",
    "Fill missing numerical values (mean)",
    "Fill missing numerical values (0)",
    "Fill missing categorical values (most frequent)",
    "Fill missing categorical values ('Unknown')",
    "Remove rows with missing values",
    "Convert column to numeric",
    "Convert column to date",
    "Convert column to text",
    "Rename column (new name below)",
    "Remove column",
    "Standardize labels (trim, lowercase, single spaces)",
]

def clean_result(op, col, new, st):
    df = need(st)["work"].copy()
    col = col or None
    m = miss_mask(df)
    if op.startswith("Run Full Auto-Clean"):
        cleaned_df, _, _ = auto_prep_pipeline(df)
        return cleaned_df
    if op.startswith("Remove fully"):
        return df.drop_duplicates().reset_index(drop=True)
    if op.startswith("Remove rows"):
        return (df[~m.any(axis=1)] if not col else df[~m[col]]).reset_index(drop=True)
    cols_need = ("Convert", "Rename", "Remove column", "Standardize")
    if op.startswith(cols_need) and not col:
        raise gr.Error("Select a target column first.")
    if op.startswith("Fill missing numerical"):
        targets = [col] if col else [c for c in df.columns if st["kinds"][c] == "num"]
        for c in targets:
            s = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False).where(~m[c]), errors="coerce")
            if s.notna().sum() == 0:
                continue
            df[c] = s.fillna(s.mean() if "mean" in op else s.median() if "median" in op else 0)
    elif op.startswith("Fill missing categorical"):
        targets = [col] if col else [c for c in df.columns if st["kinds"][c] in ("cat", "text")]
        for c in targets:
            s = df[c].where(~m[c])
            fill = "Unknown" if "Unknown" in op else (s.mode().iloc[0] if s.notna().any() else "Unknown")
            df[c] = s.fillna(fill)
    elif op.startswith("Convert column to numeric"):
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")
    elif op.startswith("Convert column to date"):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    elif op.startswith("Convert column to text"):
        df[col] = df[col].astype(str).where(df[col].notna())
    elif op.startswith("Rename"):
        if not new or not new.strip():
            raise gr.Error("Type the new column name.")
        if new in df.columns:
            raise gr.Error("A column with that name already exists.")
        df = df.rename(columns={col: new.strip()})
    elif op.startswith("Remove column"):
        df = df.drop(columns=[col])
    elif op.startswith("Standardize"):
        df[col] = df[col].astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True).where(df[col].notna())
    return df

def clean_preview(op, col, new, st):
    r = clean_result(op, col, new, st)
    w = st["work"]
    return f"🔍 **Preview:** `{len(w):,} × {w.shape[1]}` → `{len(r):,} × {r.shape[1]}` (Completeness: `{completeness(r):.1f}%`). Press **Apply** to commit.", r.head(25)

def clean_apply(op, col, new, st):
    r = clean_result(op, col, new, st)
    st["work"] = r
    st = refresh(st)
    return [
        st,
        f"✅ **Applied:** `{op}`. Working dataset is now `{len(r):,} × {r.shape[1]}` (`{completeness(r):.1f}%` complete).",
        overview(st),
        r.head(25),
        st["ready_path"],
    ] + dd(r.columns)

def restore(st):
    need(st)
    st["work"] = st["orig"].copy()
    st = refresh(st)
    return [
        st,
        "↩️ **Restored original raw dataset.**",
        overview(st),
        st["work"].head(25),
        st["ready_path"],
    ] + dd(st["work"].columns)

def export(which, fmt, st):
    need(st)
    base = os.path.join(tempfile.gettempdir(), "datapilot_" + which.split()[0].lower())
    if which.startswith("Analysis"):
        df = pd.concat([profile_table(st).assign(Section="Profile")], ignore_index=True)
    else:
        df = {
            "Original data": st["orig"],
            "Filtered data": st["work"].loc[st["fidx"]],
            "Cleaned / working data": st["work"],
        }[which]
    p = base + (".csv" if fmt == "CSV" else ".xlsx")
    if fmt == "CSV":
        df.to_csv(p, index=False)
    else:
        d2 = df.copy()
        for c in d2.columns:
            if isinstance(d2[c].dtype, pd.DatetimeTZDtype):
                d2[c] = d2[c].dt.tz_localize(None)
        d2.to_excel(p, index=False)
    return p, f"✅ Exported `{len(df):,}` rows × `{df.shape[1]}` columns to **{fmt}**."


# ---------------- Gradio UI Builder ----------------
def build_app():
    theme = gr.themes.Base(
        primary_hue="indigo",
        secondary_hue="cyan",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "sans-serif"],
    ).set(
        body_background_fill="#070913",
        block_background_fill="rgba(17, 24, 39, 0.72)",
        block_border_color="rgba(148, 163, 184, 0.16)",
        block_radius="14px",
        input_background_fill="rgba(15, 23, 42, 0.85)",
        button_primary_background_fill="linear-gradient(135deg, #6366f1, #06b6d4)",
        button_primary_text_color="#ffffff",
    )

    block_params = inspect.signature(gr.Blocks.__init__).parameters
    block_kwargs = {"title": "DataPilot AI — Conversational Analytics Studio"}
    if "theme" in block_params:
        block_kwargs["theme"] = theme
    if "css" in block_params:
        block_kwargs["css"] = CSS

    with gr.Blocks(**block_kwargs) as app:
        st = gr.State({})
        gr.HTML(f"<style>{CSS}</style>")

        gr.HTML("""
        <div class='ai-header'>
            <div class='ai-brand'>
                <div class='ai-orb'>✦</div>
                <div class='ai-title'>
                    <h1>DataPilot AI — Conversational Data Studio</h1>
                    <p>Automated Excel Hygiene • Natural Language AI Copilot • Executive Visual Dashboards</p>
                </div>
            </div>
            <div class='ai-badge'>⚡ AUTO-PREP & AI COPILOT v2.5</div>
        </div>
        """)

        with gr.Tabs():
            # ---------------- TAB 1: AI COPILOT & SMART UPLOAD ----------------
            with gr.Tab("🤖 1. AI Copilot & Smart Upload"):
                with gr.Row():
                    # Left Sidebar: Ingestion & Instant Ready-to-Use Excel
                    with gr.Column(scale=4):
                        gr.Markdown("### 📂 1. Upload & Auto-Prepare Excel / CSV")
                        f = gr.File(label="Drop Excel, CSV, TSV, or JSON File", type="filepath", file_types=[".csv", ".tsv", ".xlsx", ".xls", ".json"])
                        sheet = gr.Dropdown(label="Excel Worksheet", choices=[])
                        url = gr.Textbox(label="Or Paste Public Google Sheets URL", placeholder="https://docs.google.com/spreadsheets/d/...")
                        auto_clean = gr.Checkbox(
                            value=True,
                            label="✨ Auto-Clean & Make Analysis-Ready on Upload (dedupe, fix missing values, strip noise)"
                        )
                        load = gr.Button("🚀 Load, Auto-Clean & Launch AI Copilot", variant="primary")
                        status = gr.Markdown("No dataset loaded yet.")
                        audit_box = gr.HTML()
                        ready_excel = gr.File(label="📥 Download Analysis-Ready Excel (.xlsx)")

                    # Right Main Area: AI Chatbot Interface
                    with gr.Column(scale=8):
                        gr.Markdown("### 💬 2. Ask DataPilot AI Anything About Your Dataset")
                        welcome_text = (
                            "👋 **Welcome to DataPilot AI!**\n\n"
                            "Upload your raw **Excel (`.xlsx`/`.xls`)** or **CSV** file on the left. "
                            "I will automatically:\n"
                            "1. **Remove duplicate rows** and strip unexpected formatting symbols.\n"
                            "2. **Handle missing values** using smart statistical imputation.\n"
                            "3. **Generate a clean, ready-to-use Excel file** for immediate download.\n"
                            "4. Answer your questions with **live charts, KPIs, and tables** right here in chat!"
                        )
                        chatbot = create_chatbot(
                            format_chat_init(welcome_text),
                            height=380,
                            label="DataPilot AI Analyst"
                        )
                        with gr.Row():
                            chat_in = gr.Textbox(
                                placeholder="Ask e.g. 'Summarize this dataset', 'Show correlation heatmap', 'Find outliers', 'Plot top categories'...",
                                show_label=False,
                                scale=5
                            )
                            chat_btn = gr.Button("Send ✦", variant="primary", scale=1)

                        with gr.Row():
                            qp1 = gr.Button("📊 Executive Summary", size="sm")
                            qp2 = gr.Button("🧹 Auto-Clean Report", size="sm")
                            qp3 = gr.Button("🔗 Correlation Heatmap", size="sm")
                            qp4 = gr.Button("🔍 Detect Outliers", size="sm")
                            qp5 = gr.Button("📈 Top Category Chart", size="sm")

                        with gr.Row():
                            ai_plot = gr.Plot(label="AI Generated Visualization")
                        ai_table = gr.Dataframe(label="AI Supporting Data Table", interactive=False)

                ov = gr.HTML()
                with gr.Accordion("📋 Column Schema Profile & Sample Data Preview", open=False):
                    prof = gr.Dataframe(label="Detected Column Schema & Summary Stats", interactive=False)
                    n = gr.Radio([10, 25, 50], value=10, label="Preview rows")
                    pv = gr.Dataframe(label="Cleaned Working Dataset Preview", interactive=False)

            # ---------------- TAB 2: EXECUTIVE DASHBOARD ----------------
            with gr.Tab("📊 2. Executive Dashboard"):
                gr.Markdown("Generate a complete, multi-chart interactive dashboard and downloadable HTML report from your cleaned dataset.")
                bd = gr.Button("⚡ GENERATE EXECUTIVE DASHBOARD", variant="primary", size="lg")
                dhtml = gr.HTML()
                with gr.Row():
                    dplot1 = gr.Plot()
                    dplot2 = gr.Plot()
                with gr.Row():
                    dplot3 = gr.Plot()
                    dplot4 = gr.Plot()
                with gr.Row():
                    dplot5 = gr.Plot()
                    dplot6 = gr.Plot()
                dplots = [dplot1, dplot2, dplot3, dplot4, dplot5, dplot6]
                dtab = gr.Dataframe(label="Analyzed Records Sample (First 25)", interactive=False)
                dfile = gr.File(label="📥 Download Standalone HTML Dashboard Report")

            # ---------------- TAB 3: VISUAL ANALYTICS STUDIO ----------------
            with gr.Tab("🎨 3. Visual Chart Builder"):
                gr.Markdown("Build custom interactive Plotly charts from your cleaned and filtered dataset.")
                with gr.Row():
                    ct = gr.Dropdown(CHARTS, value="Bar chart", label="Chart Type")
                    vx = gr.Dropdown(label="X-Axis Column", choices=[])
                    vy = gr.Dropdown(label="Y-Axis Column", choices=["(none)"], value="(none)")
                with gr.Row():
                    vc = gr.Dropdown(label="Color / Group By", choices=["(none)"], value="(none)")
                    ag = gr.Dropdown(list(AGGS), value="Count", label="Aggregation")
                    tn = gr.Slider(3, 50, value=12, step=1, label="Top N Categories")
                bv = gr.Button("✨ Render Interactive Chart", variant="primary")
                vmsg = gr.Markdown()
                vplot = gr.Plot()

            # ---------------- TAB 4: DEEP EDA ----------------
            with gr.Tab("🔬 4. Automated EDA"):
                gr.Markdown("Comprehensive statistical profiling across numerical, categorical, date, and outlier dimensions.")
                be = gr.Button("🚀 Run Deep Statistical EDA", variant="primary")
                e1 = gr.Dataframe(label="Numerical Distribution Metrics (Mean, Median, Std, IQR)", interactive=False)
                e2 = gr.Dataframe(label="Categorical Dominance & Cardinality", interactive=False)
                e3 = gr.Dataframe(label="Potential Outliers (1.5 × IQR Rule)", interactive=False)
                e4 = gr.Dataframe(label="Date / Time Span Coverage", interactive=False)
                e5 = gr.Plot(label="Feature Correlation Matrix")

            # ---------------- TAB 5: DATA QUALITY INSPECTOR ----------------
            with gr.Tab("🛡️ 5. Data Quality Inspector"):
                qsum = gr.HTML()
                b0 = gr.Button("🔄 Refresh Quality Audit")
                mt = gr.Dataframe(label="Missing Values Breakdown by Column", interactive=False)
                with gr.Row():
                    qcol = gr.Dropdown(label="Inspect Column for Missing Rows (leave blank for any)", choices=[])
                    b1 = gr.Button("View Rows With Missing Values")
                mr = gr.Dataframe(interactive=False)
                dcols = gr.Dropdown(label="Columns to Compare for Duplicates", multiselect=True, choices=[])
                with gr.Row():
                    d1 = gr.Button("View Exact Duplicate Rows")
                    d2 = gr.Button("View Unique Rows")
                    d3 = gr.Button("Inspect Duplicates by Column")
                    d4 = gr.Button("Inspect Potential Duplicate Records")
                dout = gr.Dataframe(interactive=False)
                b2 = gr.Button("Scan for Data Type & Formatting Issues")
                ti = gr.Dataframe(interactive=False)

            # ---------------- TAB 6: FILTER & SLICE EXPLORER ----------------
            with gr.Tab("🔎 6. Filter & Slice Explorer"):
                gr.Markdown("Stack multiple filter conditions to slice your data. All charts, EDA, and dashboards respect active filters.")
                with gr.Row():
                    fcol = gr.Dropdown(label="Column", choices=[])
                    fop = gr.Dropdown(label="Condition", choices=TXT_OPS)
                    fv1 = gr.Textbox(label="Value / Minimum / Start Date")
                    fv2 = gr.Textbox(label="Maximum / End Date (for 'between')")
                fhint = gr.Markdown()
                with gr.Row():
                    fa = gr.Button("➕ Add & Apply Filter", variant="primary")
                    fu = gr.Button("↩️ Undo Last Filter")
                    fr = gr.Button("🗑️ Clear All Filters")
                fsum = gr.Markdown()
                fstats = gr.Dataframe(label="Numerical Summary of Filtered Subset", interactive=False)
                ftab = gr.Dataframe(label="Filtered Rows Preview (First 100)", interactive=False)

            # ---------------- TAB 7: CLEANING & EXPORT STUDIO ----------------
            with gr.Tab("🧹 7. Custom Cleaning & Export"):
                gr.Markdown("Fine-tune individual columns or restore the original raw upload at any time.")
                with gr.Row():
                    co = gr.Dropdown(CLEAN_OPS, value=CLEAN_OPS[0], label="Cleaning Operation")
                    cc = gr.Dropdown(label="Target Column (optional for global ops)", choices=[])
                    cn = gr.Textbox(label="New Column Name (for Rename)")
                with gr.Row():
                    cp = gr.Button("🔍 Preview Change")
                    ca = gr.Button("✅ Apply to Working Dataset", variant="primary")
                    crs = gr.Button("↩️ Restore Original Raw Upload")
                cmsg = gr.Markdown()
                ctab = gr.Dataframe(interactive=False)
                gr.Markdown("### 📦 Export Dataset or Summary Report")
                with gr.Row():
                    xw = gr.Dropdown(
                        ["Cleaned / working data", "Filtered data", "Original data", "Analysis summary"],
                        value="Cleaned / working data",
                        label="Dataset Version"
                    )
                    xf = gr.Radio(["XLSX", "CSV"], value="XLSX", label="Export Format")
                    bx = gr.Button("📥 Generate Export File", variant="primary")
                xfile = gr.File(label="Download Exported File")
                xmsg = gr.Markdown()

        # ---------------- Event Wiring ----------------
        dd_out = [qcol, dcols, fcol, vx, cc, vc, vy]

        load.click(
            load_data,
            [f, sheet, url, auto_clean, n, st],
            [st, status, audit_box, ov, prof, pv, ready_excel, chatbot] + dd_out
        ).then(
            lambda s: (quality_summary(s), missing_table(s)) if s and "orig" in s else ("", None),
            st,
            [qsum, mt]
        )

        f.change(on_file, f, sheet)
        n.change(preview, [n, st], pv)

        # AI Chatbot Events
        chat_btn.click(ai_copilot_reply, [chat_in, chatbot, st], [chatbot, ai_plot, ai_table]).then(lambda: "", None, chat_in)
        chat_in.submit(ai_copilot_reply, [chat_in, chatbot, st], [chatbot, ai_plot, ai_table]).then(lambda: "", None, chat_in)

        qp1.click(lambda h, s: ai_copilot_reply("Give me a complete executive summary of this dataset", h, s), [chatbot, st], [chatbot, ai_plot, ai_table])
        qp2.click(lambda h, s: ai_copilot_reply("Show me the data cleaning and missing value report", h, s), [chatbot, st], [chatbot, ai_plot, ai_table])
        qp3.click(lambda h, s: ai_copilot_reply("Show the correlation heatmap and strongest relationships", h, s), [chatbot, st], [chatbot, ai_plot, ai_table])
        qp4.click(lambda h, s: ai_copilot_reply("Find potential outliers using IQR", h, s), [chatbot, st], [chatbot, ai_plot, ai_table])
        qp5.click(lambda h, s: ai_copilot_reply("Plot a bar chart of the top categories", h, s), [chatbot, st], [chatbot, ai_plot, ai_table])

        # Quality, Explorer, EDA, Chart, Dashboard, Cleaning Events
        b0.click(lambda s: (quality_summary(s), missing_table(s)), st, [qsum, mt])
        b1.click(missing_rows, [qcol, st], mr)
        for b, mode in ((d1, "full"), (d2, "unique"), (d3, "column"), (d4, "potential")):
            b.click(lambda c, s, m=mode: dup_view(m, c, s), [dcols, st], dout)
        b2.click(type_issues, st, ti)

        fcol.change(col_changed, [fcol, st], [fop, fhint])
        fo = [st, fsum, ftab, fstats]
        fa.click(add_filter, [fcol, fop, fv1, fv2, st], fo)
        fu.click(undo_filter, st, fo)
        fr.click(reset_filters, st, fo)

        be.click(eda, st, [e1, e2, e3, e4, e5])
        vy.change(agg_options, [vy, st], ag)
        bv.click(chart_cb, [ct, vx, vy, vc, ag, tn, st], [vplot, vmsg])
        bd.click(dashboard, st, [dhtml, dtab, dfile] + dplots)

        cp.click(clean_preview, [co, cc, cn, st], [cmsg, ctab])
        co_out = [st, cmsg, ov, ctab, ready_excel] + dd_out
        ca.click(clean_apply, [co, cc, cn, st], co_out)
        crs.click(restore, st, co_out)
        bx.click(export, [xw, xf, st], [xfile, xmsg])

    return app

if __name__ == "__main__":
    build_app().queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://96d7da6e86002e7e5c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
requirements = """
pandas
numpy
gradio
scikit-learn
matplotlib
seaborn
plotly
openpyxl
xlsxwriter
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [11]:
bharatcode = """
PROJECT NAME: DataPilot AI

PROJECT CATEGORY: Work & Automation

PROJECT DESCRIPTION:

DataPilot AI is an AI-powered data analytics
application designed to simplify the data
analysis process.

It uses Python-based data analytics and
visualization technologies to help users
understand datasets and generate meaningful
insights.

TECHNOLOGIES:
Python
Pandas
NumPy
Scikit-learn
Gradio
Matplotlib
Seaborn
Plotly

AI INTEGRATION:
BharatCode API (only if actually integrated).

DEVELOPER:
Mohamed Mujeeb Chana
"""

with open("/content/BharatCode.txt", "w") as f:
    f.write(bharatcode.strip())

print("BharatCode.txt created successfully!")

BharatCode.txt created successfully!


In [12]:
readme = """
# DataPilot AI
### AI-Powered Data Analytics Platform

## About the Project

DataPilot AI is a Python-based data analytics
application designed to help users analyse
datasets and understand their data more easily.

The project uses Python data science libraries
to perform data processing, analysis and
visualization.

## Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- Gradio
- Matplotlib
- Seaborn
- Plotly

## Project Features

- Dataset analysis
- Data processing
- Statistical analysis
- Data visualization
- Graphical representation of data
- Interactive interface through Gradio

Note: The listed features should be retained
only if implemented in the actual notebook.

## Installation

Install Python dependencies:

pip install -r requirements.txt

## Running the Project

Open the DataPilot AI notebook in Google Colab
and execute its cells in the required order.

## Project Source Code

The original Google Colab notebook is included
in this GitHub repository.

## Project Objective

To make data analysis more accessible by using
Python and data science technologies to convert
datasets into meaningful information.

## Developer

Mohamed Mujeeb Chana
"""

with open("/content/README.md", "w") as f:
    f.write(readme.strip())

print("README.md created successfully!")

README.md created successfully!


In [13]:
import os

for filename in [
    "requirements.txt",
    "README.md",
    "BharatCode.txt"
]:
    path = "/content/" + filename
    print(filename, ":", "Created" if os.path.exists(path) else "Missing")

requirements.txt : Created
README.md : Created
BharatCode.txt : Created


In [14]:
import zipfile
from google.colab import files

with zipfile.ZipFile(
    "/content/DataPilot_GitHub_Files.zip", "w"
) as zipf:

    zipf.write("/content/requirements.txt", "requirements.txt")
    zipf.write("/content/README.md", "README.md")
    zipf.write("/content/BharatCode.txt", "BharatCode.txt")

files.download("/content/DataPilot_GitHub_Files.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>